In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
inputs = np.array([ [73, 67, 43],
                    [91, 88, 64],
                    [87, 134, 58],
                    [102, 43, 37],
                    [69, 96, 70],
                    [74, 66, 43],
                    [91, 87, 65],
                    [88, 134, 59],
                    [101, 44, 37],
                    [68, 96, 71],
                    [73, 66, 44],
                    [92, 87, 64],
                    [87, 135, 57],
                    [103, 43, 36],
                    [68, 97, 70] ],
                    dtype='float32')

In [3]:
targets = np.array([1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0], dtype='float32').reshape(-1, 1)

In [4]:
inputs = torch.tensor(inputs, dtype=torch.float32)
targets = torch.tensor(targets, dtype=torch.float32)

In [5]:
scaler = StandardScaler()
inputs = torch.tensor(scaler.fit_transform(inputs), dtype=torch.float32)



In [6]:
inputs

tensor([[-0.9371, -0.6076, -0.9096],
        [ 0.5340,  0.0809,  0.7466],
        [ 0.2070,  1.5890,  0.2734],
        [ 1.4330, -1.3944, -1.3828],
        [-1.2640,  0.3431,  1.2198],
        [-0.8554, -0.6404, -0.9096],
        [ 0.5340,  0.0481,  0.8254],
        [ 0.2888,  1.5890,  0.3523],
        [ 1.3512, -1.3617, -1.3828],
        [-1.3458,  0.3431,  1.2986],
        [-0.9371, -0.6404, -0.8307],
        [ 0.6157,  0.0481,  0.7466],
        [ 0.2070,  1.6217,  0.1945],
        [ 1.5147, -1.3944, -1.4616],
        [-1.3458,  0.3759,  1.2198]])

In [7]:
from torch.utils.data import TensorDataset

train_ds = TensorDataset(inputs, targets)
train_ds[0:3]

(tensor([[-0.9371, -0.6076, -0.9096],
         [ 0.5340,  0.0809,  0.7466],
         [ 0.2070,  1.5890,  0.2734]]),
 tensor([[1.],
         [1.],
         [0.]]))

In [8]:
from torch.utils.data import DataLoader

bs = 5
train_loader = DataLoader(train_ds, batch_size = bs, shuffle = True)

for xb, yb in train_loader:
    print(xb)
    print(yb)
    break

tensor([[ 0.2070,  1.6217,  0.1945],
        [ 1.3512, -1.3617, -1.3828],
        [-0.9371, -0.6076, -0.9096],
        [-0.8554, -0.6404, -0.9096],
        [-1.3458,  0.3431,  1.2986]])
tensor([[1.],
        [1.],
        [1.],
        [1.],
        [0.]])


In [9]:
class Logistic_regression(torch.nn.Module):
    def __init__(self, input_dim, output_dim):
        super(Logistic_regression, self).__init__()
        self.linear = torch.nn.Linear(input_dim, output_dim)
    def forward(self, x):
        outputs = torch.sigmoid(self.linear(x))
        return outputs

In [10]:
model = Logistic_regression(3, 1)

preds = model(inputs)

preds

tensor([[0.3552],
        [0.3471],
        [0.4933],
        [0.6286],
        [0.1602],
        [0.3616],
        [0.3346],
        [0.4916],
        [0.6222],
        [0.1493],
        [0.3426],
        [0.3534],
        [0.5072],
        [0.6478],
        [0.1565]], grad_fn=<SigmoidBackward0>)

In [11]:
criterion = nn.BCELoss()

In [12]:
opt = torch.optim.SGD(model.parameters(), lr = 1e-5)

In [20]:
def fit(num_epochs, model, loss_fn, opt, train_dl):
    total = 0
    correct = 0
    for epoch in range(num_epochs):
        epoch_loss = 0
        epoch_acc = 0
        for xb, yb in train_dl:
            pred = model(xb)
            loss = criterion(pred,yb)

            y_pred_tag = torch.round(pred)
            correct_results_sum = (y_pred_tag == yb).sum().float()
            acc = correct_results_sum/yb.shape[0]
            acc = torch.round(acc * 100)
            loss.backward()
            opt.step()
            opt.zero_grad()
            epoch_loss += loss.item()
        if(epoch+1) % 10 == 0:
            print('Epoch [{}/{}], Loss: {:.4f}, Accuracy: {:.2f}'.format(epoch+1, num_epochs, loss.item(), acc/len(train_dl)))

In [22]:
fit(100, model, criterion, opt, train_loader)

Epoch [10/100], Loss: 0.5521, Accuracy: 26.67
Epoch [20/100], Loss: 0.8410, Accuracy: 13.33
Epoch [30/100], Loss: 0.8473, Accuracy: 6.67
Epoch [40/100], Loss: 0.6742, Accuracy: 20.00
Epoch [50/100], Loss: 0.6226, Accuracy: 20.00
Epoch [60/100], Loss: 0.4222, Accuracy: 33.33
Epoch [70/100], Loss: 0.3267, Accuracy: 33.33
Epoch [80/100], Loss: 0.9018, Accuracy: 6.67
Epoch [90/100], Loss: 0.5076, Accuracy: 26.67
Epoch [100/100], Loss: 0.6296, Accuracy: 20.00


In [23]:
preds = model(inputs)
preds

tensor([[0.3553],
        [0.3472],
        [0.4934],
        [0.6288],
        [0.1602],
        [0.3617],
        [0.3347],
        [0.4917],
        [0.6223],
        [0.1493],
        [0.3427],
        [0.3535],
        [0.5073],
        [0.6480],
        [0.1565]], grad_fn=<SigmoidBackward0>)

In [24]:
model.eval()
with torch.no_grad():
    test_predictions = model(inputs)
    test_predictions = (test_predictions >= 0.5).float()

print("Predicted Labels:", test_predictions.numpy().flatten())

Predicted Labels: [0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 1. 1. 0.]


# how is logistic regression different from linear regression?


### Linear Regression
- **Type:** Regression  
- **Dependent Variable:** Continuous  
- **Purpose:** Predict numerical values  
- **Model:**  
  \[Y = \beta_0 + \beta_1 X\]
- **Output Range:** \((-\infty, +\infty)\)
- **Estimation Method:** Ordinary Least Squares (OLS)
- **Loss Function:** Mean Squared Error (MSE)
- **Example:** House price prediction, sales forecasting

---

### Logistic Regression
- **Type:** Classification  
- **Dependent Variable:** Categorical (Binary/Multinomial)  
- **Purpose:** Predict class probability  
- **Model:**  
  \[
  P(Y=1) = \frac{1}{1 + e^{-z}}, \quad z = \beta_0 + \beta_1 X
  \]
- **Output Range:** \([0, 1]\)
- **Estimation Method:** Maximum Likelihood Estimation (MLE)
- **Loss Function:** Log Loss (Cross-Entropy)
- **Example:** Spam detection, disease classification

---

### Key Differences

| Feature | Linear Regression | Logistic Regression |
|------|------------------|-------------------|
| Problem Type | Regression | Classification |
| Output | Real values | Probability (0–1) |
| Curve | Straight line | Sigmoid (S-shaped) |
| Estimation | OLS | MLE |
| Loss Function | MSE | Log Loss |

---

### When to Use
- **Linear Regression:** Predict a numerical value  
- **Logistic Regression:** Predict class or probability


# Logistic Regression and Related Concepts

## Q1: Why do we use the sigmoid activation function in Logistic Regression?

In **Logistic Regression**, we use the **sigmoid activation function** because it maps any real-valued number to a range between 0 and 1, which can be interpreted as a probability. Logistic Regression is used for binary classification tasks, where the goal is to predict a class label (0 or 1).

The sigmoid function is defined as:

\[
\sigma(z) = \frac{1}{1 + e^{-z}}
\]

Where \( z \) is the linear combination of the input features, i.e., \( z = w \cdot x + b \) (where \( w \) is the weight vector, \( x \) is the feature vector, and \( b \) is the bias term). The sigmoid function outputs a probability value between 0 and 1, which is perfect for binary classification because:

- If the output is close to 1, the model predicts class 1.
- If the output is close to 0, the model predicts class 0.

This probabilistic interpretation helps in decision-making and is essential for calculating the binary cross-entropy loss during training.

---

## Q2: What is the role of Binary Cross-Entropy Loss (BCE Loss) in training a Logistic Regression model?

**Binary Cross-Entropy Loss (BCE Loss)** is the loss function used to train a Logistic Regression model (and other binary classifiers). It measures the difference between the predicted probabilities and the actual class labels, and its purpose is to quantify how well the model's predictions align with the true labels.

The formula for BCE loss is:

\[
\text{BCE Loss} = -\left[ y \cdot \log(p) + (1 - y) \cdot \log(1 - p) \right]
\]

Where:
- \( y \) is the true label (either 0 or 1),
- \( p \) is the predicted probability from the sigmoid function.

The loss penalizes the model more when it is confident but wrong. For example:
- If the model predicts a very high probability for class 1, but the true label is 0, the loss will be large.
- Conversely, if the model predicts a very low probability for class 1, but the true label is 1, the loss will also be large.

The goal during training is to minimize this loss function by adjusting the model's parameters using optimization algorithms like **gradient descent**.

---

## Q3: What does `optimizer.zero_grad()` do, and why is it necessary in every iteration?

In PyTorch (and similar frameworks), `optimizer.zero_grad()` clears the accumulated gradients from the previous iteration of training.

- **Why is it necessary?** During backpropagation, gradients are calculated and stored for each parameter to help adjust them during the optimization process. These gradients accumulate by default in PyTorch, so if you don’t reset them, they will keep adding up across iterations, leading to incorrect updates and ultimately affecting training.

Thus, **`optimizer.zero_grad()`** ensures that the gradients for each batch start from scratch, preventing the accumulation of outdated gradients and allowing for proper weight updates in each iteration.

---

## Q4: How do we convert model outputs (probabilities) into 0 or 1 class labels?

After applying the sigmoid function, the model output is a probability between 0 and 1. To convert this into class labels (either 0 or 1), we apply a **thresholding** operation:

- If the predicted probability \( p \) is greater than or equal to 0.5, classify as **1**.
- If \( p \) is less than 0.5, classify as **0**.

This threshold of 0.5 is commonly used because it balances the decision boundary between the two classes. However, in some cases, you might adjust this threshold depending on the application (e.g., to handle class imbalance or to maximize precision or recall).

Formally, the conversion can be expressed as:

\[
\hat{y} = \begin{cases} 
1 & \text{if } p \geq 0.5 \\
0 & \text{if } p < 0.5
\end{cases}
\]

Where \( p \) is the probability output from the sigmoid function and \( \hat{y} \) is the predicted class label.

---



In [90]:
import torch

In [91]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [92]:
df = pd.read_csv(r"D:\data_ml_dl\data_dl\play_tennis.csv")

In [93]:
df

,day,outlook,temp,humidity,wind,play
0,D1,Sunny,Hot,High,Weak,No
1,D2,Sunny,Hot,High,Strong,No
2,D3,Overcast,Hot,High,Weak,Yes
3,D4,Rain,Mild,High,Weak,Yes
4,D5,Rain,Cool,Normal,Weak,Yes
5,D6,Rain,Cool,Normal,Strong,No
6,D7,Overcast,Cool,Normal,Strong,Yes
7,D8,Sunny,Mild,High,Weak,No
8,D9,Sunny,Cool,Normal,Weak,Yes
9,D10,Rain,Mild,Normal,Weak,Yes


In [94]:
df = df.drop(columns='day')

In [95]:
df

,outlook,temp,humidity,wind,play
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes


In [96]:
inputs = df.iloc[:,0:4]
inputs

,outlook,temp,humidity,wind
0,Sunny,Hot,High,Weak
1,Sunny,Hot,High,Strong
2,Overcast,Hot,High,Weak
3,Rain,Mild,High,Weak
4,Rain,Cool,Normal,Weak
5,Rain,Cool,Normal,Strong
6,Overcast,Cool,Normal,Strong
7,Sunny,Mild,High,Weak
8,Sunny,Cool,Normal,Weak
9,Rain,Mild,Normal,Weak


In [97]:
targets = df.iloc[:,4]
targets

0      No
1      No
2     Yes
3     Yes
4     Yes
5      No
6     Yes
7      No
8     Yes
9     Yes
10    Yes
11    Yes
12    Yes
13     No
Name: play, dtype: object

In [98]:
from sklearn.preprocessing import LabelEncoder

for col in inputs.columns:
    le = LabelEncoder()
    inputs[col] = le.fit_transform(inputs[col].values)
targets = targets.map({'Yes':1, 'No':0})

C:\Users\Prasad\AppData\Local\Temp\ipykernel_2392\1755492581.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inputs[col] = le.fit_transform(inputs[col].values)
C:\Users\Prasad\AppData\Local\Temp\ipykernel_2392\1755492581.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inputs[col] = le.fit_transform(inputs[col].values)
C:\Users\Prasad\AppData\Local\Temp\ipykernel_2392\1755492581.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexe

In [100]:
inputs = torch.tensor(inputs.values, dtype=torch.float32)
targets = torch.tensor(targets.values, dtype=torch.float32)

In [102]:
scaler = StandardScaler()
inputs = torch.tensor(scaler.fit_transform(inputs), dtype=torch.float32)

In [103]:
inputs

tensor([[ 1.1628, -0.1715, -1.0000,  0.8660],
        [ 1.1628, -0.1715, -1.0000, -1.1547],
        [-1.3416, -0.1715, -1.0000,  0.8660],
        [-0.0894,  1.0290, -1.0000,  0.8660],
        [-0.0894, -1.3720,  1.0000,  0.8660],
        [-0.0894, -1.3720,  1.0000, -1.1547],
        [-1.3416, -1.3720,  1.0000, -1.1547],
        [ 1.1628,  1.0290, -1.0000,  0.8660],
        [ 1.1628, -1.3720,  1.0000,  0.8660],
        [-0.0894,  1.0290,  1.0000,  0.8660],
        [ 1.1628,  1.0290,  1.0000, -1.1547],
        [-1.3416,  1.0290, -1.0000, -1.1547],
        [-1.3416, -0.1715,  1.0000,  0.8660],
        [-0.0894,  1.0290, -1.0000, -1.1547]])

In [104]:
inputs.shape, targets.shape

(torch.Size([14, 4]), torch.Size([14]))

In [105]:
inputs, targets

(tensor([[ 1.1628, -0.1715, -1.0000,  0.8660],
         [ 1.1628, -0.1715, -1.0000, -1.1547],
         [-1.3416, -0.1715, -1.0000,  0.8660],
         [-0.0894,  1.0290, -1.0000,  0.8660],
         [-0.0894, -1.3720,  1.0000,  0.8660],
         [-0.0894, -1.3720,  1.0000, -1.1547],
         [-1.3416, -1.3720,  1.0000, -1.1547],
         [ 1.1628,  1.0290, -1.0000,  0.8660],
         [ 1.1628, -1.3720,  1.0000,  0.8660],
         [-0.0894,  1.0290,  1.0000,  0.8660],
         [ 1.1628,  1.0290,  1.0000, -1.1547],
         [-1.3416,  1.0290, -1.0000, -1.1547],
         [-1.3416, -0.1715,  1.0000,  0.8660],
         [-0.0894,  1.0290, -1.0000, -1.1547]]),
 tensor([0., 0., 1., 1., 1., 0., 1., 0., 1., 1., 1., 1., 1., 0.]))

In [106]:
from torch.utils.data import TensorDataset

train_ds = TensorDataset(inputs, targets)
train_ds[0:3]

(tensor([[ 1.1628, -0.1715, -1.0000,  0.8660],
         [ 1.1628, -0.1715, -1.0000, -1.1547],
         [-1.3416, -0.1715, -1.0000,  0.8660]]),
 tensor([0., 0., 1.]))

In [107]:
from torch.utils.data import DataLoader

bs = 5
train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True)

for xb, yb in train_loader:
    print(xb)
    print(yb)
    break

tensor([[ 1.1628,  1.0290, -1.0000,  0.8660],
        [-1.3416, -0.1715,  1.0000,  0.8660],
        [ 1.1628, -0.1715, -1.0000,  0.8660],
        [-0.0894,  1.0290, -1.0000,  0.8660],
        [-0.0894,  1.0290,  1.0000,  0.8660]])
tensor([0., 1., 0., 1., 1.])


In [108]:
class Logistic_Regression(torch.nn.Module):
    def __init__(self, input_dim, output_dim):
        super(Logistic_Regression, self).__init__()
        self.linear = torch.nn.Linear(input_dim, output_dim)
    def forward(self, x):
        outputs = torch.sigmoid(self.linear(x))
        return outputs

In [109]:
xb.shape, yb.shape

(torch.Size([5, 4]), torch.Size([5]))

In [110]:
model = Logistic_Regression(4, 1 )

preds = model(inputs)

preds

tensor([[0.2659],
        [0.3338],
        [0.4637],
        [0.4405],
        [0.4654],
        [0.5463],
        [0.6504],
        [0.3376],
        [0.3604],
        [0.6328],
        [0.6068],
        [0.6272],
        [0.6543],
        [0.5213]], grad_fn=<SigmoidBackward0>)

In [111]:
import torch.nn as nn

In [112]:
criterion = nn.BCELoss()

In [113]:
model.parameters

<bound method Module.parameters of Logistic_Regression(
  (linear): Linear(in_features=4, out_features=1, bias=True)
)>

In [114]:
opt = torch.optim.SGD(model.parameters(), lr = 1e-5)

In [115]:
def fit(num_epochs, model, loss_fn, opt, train_dl):
    total = 0
    correct = 0 
    for epoch in range(num_epochs):
        epoch_loss = 0
        epoch_acc = 0
        for xb, yb in train_dl:
            yb = yb.view(-1, 1)
            pred = model(xb)
            loss = criterion(pred, yb)

            y_pred_tag = torch.round(pred)
            correct_results_sum = (y_pred_tag == yb).sum().float()
            acc = correct_results_sum/yb.shape[0]
            acc = torch.round(acc * 100)
            loss.backward()
            opt.step()
            opt.zero_grad()
            epoch_loss += loss.item()
        if (epoch+1) % 10 ==0:
            print('Epoch [{}/{}], Loss: {:,.4f}, Accuracy: {:.2f}'.format(epoch+1, num_epochs, loss.item(), acc/len(train_dl)))
            

In [116]:
fit(100, model, criterion, opt, train_loader)

Epoch [10/100], Loss: 0.5132, Accuracy: 25.00
Epoch [20/100], Loss: 0.6033, Accuracy: 16.67
Epoch [30/100], Loss: 0.6927, Accuracy: 8.33
Epoch [40/100], Loss: 0.4854, Accuracy: 25.00
Epoch [50/100], Loss: 0.4413, Accuracy: 33.33
Epoch [60/100], Loss: 0.5237, Accuracy: 25.00
Epoch [70/100], Loss: 0.6318, Accuracy: 16.67
Epoch [80/100], Loss: 0.6413, Accuracy: 16.67
Epoch [90/100], Loss: 0.5349, Accuracy: 25.00
Epoch [100/100], Loss: 0.6825, Accuracy: 8.33


In [117]:
preds = model(inputs)
preds

tensor([[0.2659],
        [0.3336],
        [0.4639],
        [0.4406],
        [0.4658],
        [0.5465],
        [0.6507],
        [0.3375],
        [0.3607],
        [0.6331],
        [0.6067],
        [0.6272],
        [0.6547],
        [0.5211]], grad_fn=<SigmoidBackward0>)

In [118]:
model.eval()
with torch.no_grad():
    test_predictions = model(inputs)
    test_predictions = (test_predictions >= 0.5).float()

print("Predicted Labels", test_predictions.numpy().flatten())

Predicted Labels [0. 0. 0. 0. 0. 1. 1. 0. 0. 1. 1. 1. 1. 1.]


In [119]:
targets

tensor([0., 0., 1., 1., 1., 0., 1., 0., 1., 1., 1., 1., 1., 0.])